# Spam V: A fair comparison of classical models

So far: Naive Bayes (counts, TF-IDF) and logistic regression. Which family should we use? A fair comparison needs

* the **same features** and the **same splits** for every model;
* the vectoriser **inside** the cross-validation loop (otherwise the vocabulary and the IDF values leak information from the
  validation folds);
* more than one number: mean and spread over folds, plus a metric that respects the class imbalance (**MCC**, **PR-AUC**);
* a look at the *cost* (training and prediction time).

We add two feature families: **word 1-2 grams** and **character 2-5 grams**. Character n-grams see *inside* words, which will
matter when the attacker obfuscates them (notebook 07).

In [ ]:
import time

from matplotlib import pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import PrecisionRecallDisplay
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

import spamlib as sl

plt.rcParams["figure.dpi"] = 100

texts, y = sl.load()
X_train, X_test, y_train, y_test = sl.split(texts, y)


def words():
    return TfidfVectorizer(
        tokenizer=sl.tokenize, token_pattern=None, lowercase=False, ngram_range=(1, 2), min_df=2, sublinear_tf=True
    )


def chars():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, sublinear_tf=True, lowercase=True)

## 1. Models

| model | idea | expected behaviour on sparse text |
|:--|:--|:--|
| Multinomial NB | independent word likelihoods | fast, strong baseline |
| Logistic regression | linear score, log-loss | strong, calibrated-ish |
| Linear SVM | linear score, maximal margin | often the best on text |
| Random forest | many decision trees | weak on sparse high-dimensional text |
| k-NN (cosine) | vote of the closest messages | memorises, slow at prediction time |
| MLP | one hidden layer | can match linear models, needs tuning |

In [ ]:
def models():
    return {
        "MultinomialNB": MultinomialNB(alpha=0.1),
        "LogisticRegression": LogisticRegression(C=30, max_iter=2000),
        "LinearSVC": LinearSVC(C=1.0),
        "RandomForest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=sl.SEED),
        "kNN (k=5, cosine)": KNeighborsClassifier(n_neighbors=5, metric="cosine", algorithm="brute"),
        "MLP (128)": MLPClassifier(hidden_layer_sizes=(128,), max_iter=300, early_stopping=True, random_state=sl.SEED),
    }

## 2. Stratified 5-fold cross-validation (on the training set only)

The test set stays untouched until the very end.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=sl.SEED)
scoring = {"f1": "f1", "precision": "precision", "recall": "recall"}

table = {}
for feat_name, feat in (("words 1-2", words), ("chars 2-5", chars)):
    for name, model in models().items():
        pipe = make_pipeline(feat(), model)
        t0 = time.time()
        res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
        table[(feat_name, name)] = {k: (res[f"test_{k}"].mean(), res[f"test_{k}"].std()) for k in scoring} | {
            "sec": time.time() - t0
        }

print(f"{'features':11} {'model':20} {'F1':>14} {'precision':>14} {'recall':>14} {'time':>7}")
for (feat_name, name), r in table.items():
    print(
        f"{feat_name:11} {name:20} "
        + " ".join(f"{r[k][0]:.3f}±{r[k][1]:.3f}".rjust(14) for k in ("f1", "precision", "recall"))
        + f" {r['sec']:6.1f}s"
    )

**Reading the table.** Differences of a few thousandths between two models are *inside the fold-to-fold noise* (the ±).
Do not declare a winner without a paired test (see `00-intro/notebook_08`, McNemar), and do not tune on the test set.

## 3. Final held-out test and precision-recall curves

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
final = {}
for name in ("MultinomialNB", "LogisticRegression", "LinearSVC"):
    for feat_name, feat in (("words", words), ("chars", chars)):
        m = make_pipeline(feat(), models()[name]).fit(X_train, y_train)
        score = (
            m.decision_function(X_test)
            if hasattr(m, "decision_function")
            else m.predict_log_proba(X_test)[:, 1] - m.predict_log_proba(X_test)[:, 0]
        )
        final[f"{name} / {feat_name}"] = sl.scores(y_test, m.predict(X_test), score)
        PrecisionRecallDisplay.from_predictions(
            y_test, score, name=f"{name} / {feat_name}", ax=ax, plot_chance_level=False
        )
ax.set_xlim(0.5, 1.0)
ax.set_ylim(0.7, 1.01)
ax.legend(fontsize=7, loc="lower left")
plt.tight_layout()
plt.show()
sl.show(final)

## 4. Take-aways

* On sparse text, **linear models on TF-IDF** are extremely hard to beat; deep models must earn their extra cost.
* Random forests and k-NN struggle with high-dimensional sparse vectors; k-NN also *stores the training set*, which is a
  liability if that data is sensitive.
* Character n-grams give a similar score with a *much larger* feature space. The reason to use them is **robustness**, not
  accuracy: see notebook 07.
* Always report a spread, not a single number.

## Exercises

1. Add `class_weight="balanced"` to LR and SVM. Which side of the precision/recall trade-off moves?
2. Combine word and character features with `sklearn.pipeline.FeatureUnion`. Is the gain larger than the fold noise?
3. Time the *prediction* of each model on the whole test set. Which one would you deploy on a mail gateway processing
   10,000 messages per second?